# Supplementary Table S7 with 95% CIs

This notebook reconstructs the current `Supplementary Table S7` from the per-cycle PR AUC exports stored under `analysis_exports/Feature_augmentation_experiment/`.

For each model, it:

- loads all per-cycle PR AUC values,
- ranks feature coalitions by mean PR AUC,
- keeps the top 3 coalitions,
- computes a two-sided 95% t-based confidence interval across cross-validation cycles,
- formats the result for direct use in the SI table.


In [ ]:
from pathlib import Path
import math

import pandas as pd
from scipy import stats

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "analysis_exports").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

BASE_DIR = PROJECT_ROOT / "analysis_exports" / "Feature_augmentation_experiment"
OUT_DIR = PROJECT_ROOT / "analysis_exports" / "supplementary_stats"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_MAP = {
    "pytorch_mlp": "PyTorch-MLP",
    "tabm": "TabM",
    "lightgbm": "LightGBM",
    "mlp": "MLP",
    "rf": "RF",
    "xgboost": "XGBoost",
}

BASE_DIR

In [ ]:
rows = []

for model_key, model_label in MODEL_MAP.items():
    path = BASE_DIR / model_key / f"{model_key}_per_cycle_all_metrics.csv"
    df = pd.read_csv(path)

    top3 = (
        df.groupby("var_tag", as_index=False)["pr_auc"]
        .mean()
        .sort_values("pr_auc", ascending=False)
        .head(3)
    )

    for coalition in top3["var_tag"]:
        vals = df.loc[df["var_tag"] == coalition, "pr_auc"].dropna().to_numpy()
        n = len(vals)
        mean = float(vals.mean())
        sd = float(vals.std(ddof=1))
        se = sd / math.sqrt(n)
        t_crit = float(stats.t.ppf(0.975, n - 1))
        ci_low = mean - t_crit * se
        ci_high = mean + t_crit * se

        rows.append(
            {
                "Model": model_label,
                "Coalition": coalition,
                "Mean PR AUC": mean,
                "SD": sd,
                "n cycles": n,
                "95% CI low": ci_low,
                "95% CI high": ci_high,
                "95% CI": f"[{ci_low:.4f}, {ci_high:.4f}]",
            }
        )

s7_df = pd.DataFrame(rows)
s7_df

In [ ]:
display_df = s7_df.copy()
display_df["Mean PR AUC"] = display_df["Mean PR AUC"].map(lambda x: f"{x:.4f}")
display_df[["Model", "Coalition", "Mean PR AUC", "95% CI"]]

In [ ]:
out_path = OUT_DIR / "table_s7_pr_auc_with_ci.csv"
s7_df.to_csv(out_path, index=False)
out_path